# Elastic Net Phase 2: macro-gain isolation (no-macro EN, VIX ablation, QLIKE)

Duplicated from `elastic_net_midterm1.ipynb` (kept as-is for midterm reference) and
extended to answer the professor's Phase 2 feedback: EN alone can't show *where* any
gain over GARCH comes from (the macro data, or just persistence-modelling at monthly
frequency). This notebook adds two more Elastic Net variants run through the exact
same walk-forward engine, so all three results are directly comparable:

- **`en_full_macro`** — full model, 11 macro vars + own-lag (unchanged from midterm1).
- **`en_no_vix`** — same, minus `VIXCLS` (implied vol is the market's own forecast,
  not a Chen-Roll-Ross macro channel — isolates "implied vol helps" from "macro
  helps").
- **`en_no_macro`** — Elastic Net with **no macro variables at all**, own-lag only.
  The benchmark `en_full_macro`'s macro-gain claim is actually measured against.

All three are Elastic Net (same `ElasticNetCV` hyperparameter search + regularized
fit), just with different feature sets — isolates the macro-information question
without also changing estimation method between the full model and the benchmark.

All three keep the same expanding-window walk-forward design as midterm1, but now
refit **every month** (not yearly) — midterm1 used a yearly freeze only because of
that deadline's time constraint; for Phase 2, GARCH/GARCH-MIDAS are also refitting
monthly, so matching that cadence keeps the comparison fair. Outputs now also include
an out-of-fold training-window residual variance (`sigma2`) and a log-normal corrected
level forecast (`fc_level`), so MSE (log scale) and QLIKE (level scale) can both be
reported.

# What this notebook does

Forecasts next-month log realized variance (`log RV²`) for 5 cyclical sector ETFs
(XLI, XLB, XLE, XLY, XLF), and runs **three Elastic Net variants** through the same
walk-forward engine so they're directly comparable: **`en_full_macro`**,
**`en_no_vix`**, and **`en_no_macro`**. Produces the common-basis outputs (log
forecast + sigma2 + log-normal corrected level forecast) and an MSE/QLIKE comparison
table across all three — the actual answer to "does macro information help, and is it
just VIX?".

**Four changes made in this version vs. midterm1** (see Phase 2 feedback + investigation notes):
1. **Price data**: now reads `daily_returns_panel_ver2.csv`, which has XLRE correctly
   spliced into XLF across the Sept 2016 spin-out (midterm1's file predates that fix).
2. **Return convention**: RV² is now built from **excess returns** (`xret_*`), matching
   the proposal's "squared daily excess returns" wording and staying consistent with
   the downstream (excess-return-based) portfolio step.
3. **Macro set**: locked to the **11 vars named in the proposal** (`T10Y2Y, DGS3MO,
   BAA10Y, CPIAUCSL, T5YIE, T5YIE_MINUS_T10YIE, VIXCLS, INDPRO, UNRATE, UMCSENT,
   DCOILWTICO`) — two more variables were added to `final_macro_data.csv` after the
   proposal was reviewed; this notebook deliberately stays on what was submitted.
4. **Refit cadence**: monthly, not yearly. Midterm1 froze hyperparameters+coefficients
   for 12 months at a time purely because of that deadline's time constraint. For
   Phase 2, GARCH/GARCH-MIDAS are refitting monthly too, so EN now matches that.

**Walk-forward mechanics**: burn-in 60 months, expanding window, refit (hyperparameters
+ coefficients together) every month. (Midterm1's rolling-window side experiment was
dropped here — expanding only.)

**Not in this notebook** (flagged for the team, not built here): portfolio weights,
Diebold-Mariano tests against GARCH/XGBoost (need their forecast files, which don't
exist in this repo yet), pooled EN with ETF dummies (depends on the XGBoost
teammate's pooling decision). The tidy output format here is shaped to make a later
merge with those easy, once they exist.

In [1]:
import os
import time
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.linear_model import ElasticNetCV, ElasticNet, LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.exceptions import ConvergenceWarning

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

PROJECT_ROOT = "../.."
CYCLICAL_TICKERS = ["XLI", "XLB", "XLE", "XLY", "XLF"]

MACRO_PATH = os.path.join(PROJECT_ROOT, "dataset", "macro_variables", "final_clean", "final_macro_data.csv")
# ver2 has XLRE correctly spliced into XLF across the Sept 2016 spin-out (see title cell)
PRICE_PATH = os.path.join(PROJECT_ROOT, "dataset", "price", "clean", "daily_returns_panel_ver2.csv")
FORECASTS_DIR = os.path.join(PROJECT_ROOT, "forecasts")  # top-level, not nested under dataset/

BURN_IN = 60       # months of history required before the first fit/forecast
REFIT_EVERY = 1    # months between hyperparameter search + coefficient refits -- monthly
                    # for Phase 2 (GARCH is also refitting monthly now, so cadence stays
                    # matched; midterm1 used 12 only because of the time constraint then)
L1_RATIOS = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0]

## Data

Macro predictors from `final_macro_data.csv`, restricted to the 11 proposal-specified
variables (already vintage/publication-lag corrected, transformed level-vs-log-diff
per channel upstream — no re-derivation needed here). Target built from CRSP `DlyRet`
converted to **excess returns** (`xret_*` = raw return minus risk-free rate) -> monthly
RV² (sum of squared daily excess returns) -> `log(RV²)`, aligned so `X.loc[t]` (known
at month t) pairs with `y.loc[t] = log(RV²)` realized in month t+1.

In [2]:
# Explicit allowlist (not "all columns minus NBUSBIS" -- that was dead code, since NBUSBIS
# is already gone upstream, and would have silently pulled in DEXUSEU/USEPUINDXD, which
# were added to final_macro_data.csv *after* the proposal's 11-variable set was reviewed).
FINAL_MACRO_VARS = [
    "T10Y2Y", "DGS3MO", "BAA10Y", "CPIAUCSL", "T5YIE", "T5YIE_MINUS_T10YIE",
    "VIXCLS", "INDPRO", "UNRATE", "UMCSENT", "DCOILWTICO",
]

macro = pd.read_csv(MACRO_PATH, index_col="date", parse_dates=True)
macro = macro[FINAL_MACRO_VARS].copy()
print(f"{len(FINAL_MACRO_VARS)} macro variables (proposal set, vintage/publication-lag corrected):", FINAL_MACRO_VARS)
print(f"macro range: {macro.index.min().date()} to {macro.index.max().date()}\n")

price = pd.read_csv(PRICE_PATH, index_col="date", parse_dates=True)
# xret_ (excess returns) per proposal Sec.2 -- keeps RV2 consistent with the downstream
# excess-return-based portfolio, instead of midterm1's raw ret_ columns.
xret_cols = {ticker: f"xret_{ticker}" for ticker in CYCLICAL_TICKERS}
daily_returns = price[list(xret_cols.values())].rename(columns={v: k for k, v in xret_cols.items()})

monthly_rv_squared = daily_returns.pow(2).resample("ME").sum()
log_rv_squared = np.log(monthly_rv_squared)
print("log_rv_squared:", log_rv_squared.index.min().date(), "to", log_rv_squared.index.max().date(),
      f"({len(log_rv_squared)} months)")

# daily_returns_panel_ver2.csv truncates to SAMPLE_START_DATE="2003-01-31" (a hard cutoff,
# not an aggregation) -- so January 2003's daily history (Jan 2-30) is dropped and only
# Jan 31 survives. The resample("ME").sum() above therefore builds Jan 2003's RV2 from a
# single day instead of a full month -- an artificially tiny, non-representative value.
# Drop that one contaminated month before building aligned_data (every month from Feb 2003
# onward has complete daily history, so this is the only affected row per ticker).
log_rv_squared = log_rv_squared.iloc[1:]

aligned_data = {}
for ticker in CYCLICAL_TICKERS:
    own_lag = log_rv_squared[ticker].rename("own_log_rv_sq_lag0")
    y = log_rv_squared[ticker].shift(-1)
    y.name = "y_log_rv_squared_tplus1"
    df = macro.join(own_lag, how="inner").join(y, how="inner").dropna()
    aligned_data[ticker] = df
    print(f"{ticker}: {df.shape[0]} aligned rows, {df.index.min().date()} to {df.index.max().date()}")

ALL_FEATURE_COLS = FINAL_MACRO_VARS + ["own_log_rv_sq_lag0"]
print()
print(f"{len(ALL_FEATURE_COLS)} features (full EN model):", ALL_FEATURE_COLS)

11 macro variables (proposal set, vintage/publication-lag corrected): ['T10Y2Y', 'DGS3MO', 'BAA10Y', 'CPIAUCSL', 'T5YIE', 'T5YIE_MINUS_T10YIE', 'VIXCLS', 'INDPRO', 'UNRATE', 'UMCSENT', 'DCOILWTICO']
macro range: 2003-01-31 to 2026-07-31

log_rv_squared: 2003-01-31 to 2025-12-31 (276 months)
XLI: 274 aligned rows, 2003-02-28 to 2025-11-30
XLB: 274 aligned rows, 2003-02-28 to 2025-11-30
XLE: 274 aligned rows, 2003-02-28 to 2025-11-30
XLY: 274 aligned rows, 2003-02-28 to 2025-11-30
XLF: 274 aligned rows, 2003-02-28 to 2025-11-30

12 features (full EN model): ['T10Y2Y', 'DGS3MO', 'BAA10Y', 'CPIAUCSL', 'T5YIE', 'T5YIE_MINUS_T10YIE', 'VIXCLS', 'INDPRO', 'UNRATE', 'UMCSENT', 'DCOILWTICO', 'own_log_rv_sq_lag0']


## Three Elastic Net variants, one shared engine

- **`en_full_macro`** — all 11 macro vars + own-lag (12 features, see
  `ALL_FEATURE_COLS` above).
- **`en_no_vix`** — same minus `VIXCLS` (10 macro vars + own-lag). VIX is
  option-implied volatility -- the market's own vol forecast, not a macro channel in
  the Chen-Roll-Ross sense. If `en_full_macro`'s edge over `en_no_macro` disappears
  here, the finding is "implied vol helps" rather than "macro information helps".
- **`en_no_macro`** — Elastic Net with own-lag only, no macro variables at all. This
  is the benchmark that answers the actual Phase 2 question: does `en_full_macro`'s
  macro information add anything over a monthly persistence model on its own? Still
  Elastic Net (same `ElasticNetCV` search), just with a single-feature set -- keeps
  the estimation method identical across all three, so only the feature set differs.

All three run through the identical walk-forward loop below (same burn-in, same
monthly refit cadence) so differences in the results are attributable to the feature
set, not to any difference in how the window, refit timing, or model class works.

In [3]:
FEATURE_SETS = {
    "en_full_macro": ALL_FEATURE_COLS,
    "en_no_vix": [c for c in ALL_FEATURE_COLS if c != "VIXCLS"],
    "en_no_macro": ["own_log_rv_sq_lag0"],
}
# all three are Elastic Net -- same ElasticNetCV hyperparameter search, only the
# feature set differs between them (see markdown above)
MODEL_KINDS = {name: "elasticnet" for name in FEATURE_SETS}

for name, cols in FEATURE_SETS.items():
    print(f"{name:14s}: {len(cols)} features -> {cols}")

en_full_macro : 12 features -> ['T10Y2Y', 'DGS3MO', 'BAA10Y', 'CPIAUCSL', 'T5YIE', 'T5YIE_MINUS_T10YIE', 'VIXCLS', 'INDPRO', 'UNRATE', 'UMCSENT', 'DCOILWTICO', 'own_log_rv_sq_lag0']
en_no_vix     : 11 features -> ['T10Y2Y', 'DGS3MO', 'BAA10Y', 'CPIAUCSL', 'T5YIE', 'T5YIE_MINUS_T10YIE', 'INDPRO', 'UNRATE', 'UMCSENT', 'DCOILWTICO', 'own_log_rv_sq_lag0']
en_no_macro   : 1 features -> ['own_log_rv_sq_lag0']


## Walk-forward engine — shared by all 3 models, expanding window, monthly refit

Same mechanics as midterm1, generalized to handle all three model kinds in one
function (the function also supports a `window` argument for a rolling-window
variant, unused here since this notebook is expanding-only) -- but now with
`REFIT_EVERY = 1`, i.e. refitting every month instead of midterm1's yearly freeze
(GARCH/GARCH-MIDAS are also refitting monthly for Phase 2, so this keeps the cadence
matched -- see title cell):

- At each `REFIT_EVERY`-month cycle: for `elasticnet`, `ElasticNetCV` (5-fold
  `TimeSeriesSplit`) picks `alpha`/`l1_ratio` on the training window (`X[0:i]`, grows
  every cycle), then a plain `ElasticNet` is fit with those; for `ols`, just
  `LinearRegression`.
- **Out-of-fold `sigma2`.** At the same refit cycle, manually loop the *same*
  `TimeSeriesSplit(5)` splits (used for hyperparameter selection) to get genuinely
  held-out predictions over the training window -- no extra look-ahead. (Not
  `cross_val_predict`: `TimeSeriesSplit` never tests its smallest initial chunk, so the
  test folds aren't a full partition, which `cross_val_predict` requires.)
  `sigma2 = var(actual - out-of-fold pred)` over whichever rows did get tested. This
  avoids using in-sample refit residuals, which understate error for a regularized fit.
- **Log-normal corrected level forecast.** `fc_level = exp(fc_log + 0.5*sigma2)` --
  plain `exp()` of a log forecast is biased low as a variance estimate; this corrects
  for that using the training-window residual variance.

In [4]:
def walk_forward_yearly(df, feature_cols, model_kind, window=None,
                         burn_in=BURN_IN, refit_every=REFIT_EVERY, l1_ratios=L1_RATIOS):
    """Shared walk-forward engine for en_full_macro / en_no_vix / en_no_macro,
    expanding or rolling. (Name kept from midterm1; refit_every=1 now, not yearly.)

    window=None -> expanding (train on X[:i]); window=60 -> rolling (train on
    X[i-window:i]). At each refit_every-month cycle: pick hyperparameters (elasticnet)
    or just fit (ols -- supported for generality, unused in this notebook since all 3
    variants here are Elastic Net), refit the final model, scaler frozen with it, and
    compute out-of-fold sigma2 by manually looping the same TimeSeriesSplit(5) used for
    hyperparameter selection -- all frozen until the next cycle. Returns one row per
    forecast month: log forecast, sigma2, log-normal corrected level forecast, plus
    actuals and hyperparameters used.
    """
    X = df[feature_cols].to_numpy()
    y = df["y_log_rv_squared_tplus1"].to_numpy()
    dates = df.index
    n = len(df)
    start = burn_in if window is None else window
    rows = []
    pipe = None
    sigma2 = np.nan
    alpha_ = l1_ratio_ = np.nan

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ConvergenceWarning)
        for i in range(start, n):
            train_start = 0 if window is None else i - window
            refit_this_month = (i - start) % refit_every == 0
            if refit_this_month:
                X_train, y_train = X[train_start:i], y[train_start:i]
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train)

                if model_kind == "elasticnet":
                    cv_search = ElasticNetCV(
                        l1_ratio=l1_ratios, cv=TimeSeriesSplit(n_splits=5),
                        max_iter=10000, random_state=0,
                    )
                    cv_search.fit(X_train_scaled, y_train)
                    alpha_, l1_ratio_ = cv_search.alpha_, cv_search.l1_ratio_
                    model = ElasticNet(alpha=alpha_, l1_ratio=l1_ratio_, max_iter=10000, random_state=0)
                else:  # "ols" -- no hyperparameters to search
                    alpha_ = l1_ratio_ = np.nan
                    model = LinearRegression()
                model.fit(X_train_scaled, y_train)

                # Out-of-fold residuals on the SAME training window + CV splits used for
                # hyperparameter selection -- no extra look-ahead, avoids the
                # in-sample-understates-error problem of a regularized fit. TimeSeriesSplit
                # never tests its very first (smallest) training chunk, so this isn't a full
                # partition -- loop the splits manually instead of sklearn's
                # cross_val_predict (which requires one), and only use whichever rows
                # actually got tested.
                oof_pred = np.full(len(y_train), np.nan)
                for tr_idx, te_idx in TimeSeriesSplit(n_splits=5).split(X_train_scaled):
                    fold_model = clone(model)
                    fold_model.fit(X_train_scaled[tr_idx], y_train[tr_idx])
                    oof_pred[te_idx] = fold_model.predict(X_train_scaled[te_idx])
                tested = ~np.isnan(oof_pred)
                sigma2 = float(np.var(y_train[tested] - oof_pred[tested]))

                # frozen scaler + frozen model, wrapped so .predict() applies both in order
                pipe = Pipeline([("scaler", scaler), ("model", model)])

            pred_log = pipe.predict(X[i:i + 1])[0]
            pred_level = np.exp(pred_log + 0.5 * sigma2)  # log-normal correction
            rows.append({
                "date": dates[i],
                "actual_logRV": y[i],
                "actual_RV": np.exp(y[i]),
                "fc_log": pred_log,
                "sigma2": sigma2,
                "fc_level": pred_level,
                "alpha": alpha_,
                "l1_ratio": l1_ratio_,
                "refit_this_month": refit_this_month,
                "train_months": i - train_start,
            })
    return pd.DataFrame(rows).set_index("date")

In [5]:
def run_all_models(window, label):
    """Runs en_full_macro / en_no_vix / en_no_macro for all 5 tickers at the given
    window setting (None=expanding, 60=rolling). Returns (wide_by_model, tidy_df):
      - wide_by_model[model][ticker] = per-ticker forecast frame (for accuracy_table)
      - tidy_df = one long table, all models/tickers/months stacked -- the common-basis
        output meant to be merged later with GARCH/GARCH-MIDAS/XGB forecasts.
    """
    wide_by_model = {}
    tidy_rows = []
    t0 = time.time()
    for model_name, feature_cols in FEATURE_SETS.items():
        wide_by_model[model_name] = {}
        for ticker in CYCLICAL_TICKERS:
            df = walk_forward_yearly(
                aligned_data[ticker], feature_cols, MODEL_KINDS[model_name], window=window
            )
            wide_by_model[model_name][ticker] = df
            tidy = df.reset_index()[["date", "actual_logRV", "actual_RV", "fc_log", "sigma2", "fc_level"]].copy()
            tidy.insert(0, "etf", ticker)
            tidy.insert(0, "model", model_name)
            tidy_rows.append(tidy)
        print(f"{model_name:14s}: {len(wide_by_model[model_name]['XLI'])} forecasts/ticker, "
              f"{time.time() - t0:.1f}s elapsed")
    tidy_df = pd.concat(tidy_rows, ignore_index=True)

    os.makedirs(FORECASTS_DIR, exist_ok=True)
    out_path = os.path.join(FORECASTS_DIR, f"en_forecasts_{label}_tidy.csv")
    tidy_df.to_csv(out_path, index=False)
    print(f"Saved {out_path} ({len(tidy_df)} rows = {len(FEATURE_SETS)} models x "
          f"{len(CYCLICAL_TICKERS)} tickers x {len(wide_by_model['en_full_macro']['XLI'])} months)")
    return wide_by_model, tidy_df


forecasts_expanding_by_model, tidy_expanding_df = run_all_models(window=None, label="expanding")

en_full_macro : 214 forecasts/ticker, 157.3s elapsed
en_no_vix     : 214 forecasts/ticker, 318.9s elapsed
en_no_macro   : 214 forecasts/ticker, 456.3s elapsed
Saved ../../forecasts/en_forecasts_expanding_tidy.csv (3210 rows = 3 models x 5 tickers x 214 months)


## Accuracy (log scale) — diagnostic, per model

Same MSE/RMSE/MAE + R²_OOS (vs. leak-free expanding historical mean, and vs.
persistence) as midterm1, now looped over all 3 models instead of just EN. Useful as a
mid-notebook sanity check; the table that actually answers the Phase 2 question (MSE +
QLIKE, EN vs EN-noVIX vs AR1, level-scale QLIKE included) is further down.

In [6]:
def historical_mean_benchmark(aligned_df, start_idx):
    y_full = aligned_df["y_log_rv_squared_tplus1"]
    n = len(y_full)
    benchmark = [y_full.iloc[:i].mean() for i in range(start_idx, n)]
    return pd.Series(benchmark, index=y_full.index[start_idx:n])


def accuracy_table(forecasts, aligned, start_idx, label):
    """forecasts: dict ticker -> walk_forward_yearly() output (actual_logRV/fc_log cols)."""
    rows, persistence_rows = [], []
    for ticker in CYCLICAL_TICKERS:
        df = forecasts[ticker]
        actual = df["actual_logRV"]
        pred = df["fc_log"]

        benchmark = historical_mean_benchmark(aligned[ticker], start_idx).reindex(df.index)
        resid = actual - pred
        mse = (resid ** 2).mean()
        r2_oos = 1 - (resid ** 2).sum() / ((actual - benchmark) ** 2).sum()
        rows.append({
            "ticker": ticker, "n_forecasts": len(df),
            "MSE": mse, "RMSE": np.sqrt(mse), "MAE": resid.abs().mean(), "R2_OOS": r2_oos,
        })

        persistence_pred = actual.shift(1)
        valid = persistence_pred.notna()
        resid_model = (actual - pred)[valid]
        resid_naive = (actual - persistence_pred)[valid]
        r2_vs_persistence = 1 - (resid_model ** 2).sum() / (resid_naive ** 2).sum()
        persistence_rows.append({
            "ticker": ticker,
            "MSE_model": (resid_model ** 2).mean(),
            "MSE_persistence": (resid_naive ** 2).mean(),
            "R2_OOS_vs_persistence": r2_vs_persistence,
        })

    acc_df = pd.DataFrame(rows).set_index("ticker")
    pers_df = pd.DataFrame(persistence_rows).set_index("ticker")
    print(f"=== {label}: accuracy ===")
    print(acc_df)
    print()
    print(f"=== {label}: vs persistence baseline ===")
    print(pers_df)
    return acc_df, pers_df


accuracy_expanding = {}
for model_name in FEATURE_SETS:
    accuracy_expanding[model_name] = accuracy_table(
        forecasts_expanding_by_model[model_name], aligned_data, BURN_IN,
        f"Expanding window, {model_name}"
    )

=== Expanding window, en_full_macro: accuracy ===
        n_forecasts       MSE      RMSE       MAE    R2_OOS
ticker                                                     
XLI             214  0.455388  0.674824  0.482421  0.499264
XLB             214  0.420332  0.648330  0.478592  0.498115
XLE             214  0.469333  0.685079  0.493816  0.500472
XLY             214  0.491828  0.701305  0.536920  0.532074
XLF             214  0.613908  0.783523  0.592812  0.572537

=== Expanding window, en_full_macro: vs persistence baseline ===
        MSE_model  MSE_persistence  R2_OOS_vs_persistence
ticker                                                   
XLI      0.452702         0.629000               0.280282
XLB      0.417680         0.536835               0.221959
XLE      0.466707         0.513143               0.090492
XLY      0.493639         0.578056               0.146036
XLF      0.611645         0.755288               0.190183
=== Expanding window, en_no_vix: accuracy ===
        n_fo

## MSE (log RV) + QLIKE (RV level) — the table that answers the Phase 2 question

Two losses, each on the scale they're meaningful for:
- **`MSE_logRV`** = `mean((actual_logRV - fc_log)^2)` — same log-scale error used
  throughout this notebook so far.
- **`QLIKE`** = `RV/RV_hat - log(RV/RV_hat) - 1`, computed on the **level** scale using
  the log-normal corrected `fc_level` (not naive `exp(fc_log)`). QLIKE is robust to
  noise in the RV proxy (MSE on log RV isn't), and it's the loss that matches what the
  portfolio step actually uses (levels, not logs) — per the Phase 2 feedback, "two
  losses are enough."

**How to read the table:** compare `en_no_macro` against `en_full_macro` row-by-row —
if `en_full_macro` is clearly better, that's evidence the 11 macro variables add real
value beyond monthly persistence; if they're close, `en_full_macro`'s apparent edge
(if any) over GARCH is probably coming from persistence-modelling at monthly
frequency, not the macro data. Compare `en_full_macro` against `en_no_vix` the same
way for the VIX-specific question: if the edge disappears without VIX, the finding is
"implied vol helps" rather than "macro information helps". `pooled` rows combine all
5 tickers' residuals into one score, per model.

In [7]:
def qlike(actual_rv, fc_level):
    ratio = actual_rv / fc_level
    return ratio - np.log(ratio) - 1


def mse_qlike_table(tidy_df, label):
    """Returns (mse_wide, qlike_wide): rows = model, columns = ticker (+ pooled) --
    read down a column to compare models on the same ticker."""
    tidy_df = tidy_df.copy()
    tidy_df["sq_err_log"] = (tidy_df["actual_logRV"] - tidy_df["fc_log"]) ** 2
    tidy_df["qlike"] = qlike(tidy_df["actual_RV"], tidy_df["fc_level"])

    per_ticker = (
        tidy_df.groupby(["model", "etf"])
        .agg(MSE_logRV=("sq_err_log", "mean"), QLIKE=("qlike", "mean"))
        .reset_index()
    )
    pooled = (
        tidy_df.groupby("model")
        .agg(MSE_logRV=("sq_err_log", "mean"), QLIKE=("qlike", "mean"))
        .reset_index()
    )
    pooled["etf"] = "pooled"
    long = pd.concat([per_ticker, pooled], ignore_index=True)

    model_order = list(FEATURE_SETS)
    ticker_order = CYCLICAL_TICKERS + ["pooled"]
    mse_wide = long.pivot(index="model", columns="etf", values="MSE_logRV").reindex(index=model_order, columns=ticker_order)
    qlike_wide = long.pivot(index="model", columns="etf", values="QLIKE").reindex(index=model_order, columns=ticker_order)

    print(f"=== {label}: MSE (log RV), model x ticker ===")
    print(mse_wide)
    print()
    print(f"=== {label}: QLIKE (RV level), model x ticker ===")
    print(qlike_wide)
    return mse_wide, qlike_wide


mse_expanding, qlike_expanding = mse_qlike_table(tidy_expanding_df, "Expanding window")

=== Expanding window: MSE (log RV), model x ticker ===
etf                 XLI       XLB       XLE       XLY       XLF    pooled
model                                                                    
en_full_macro  0.455388  0.420332  0.469333  0.491828  0.613908  0.490158
en_no_vix      0.528692  0.479018  0.481542  0.525842  0.670305  0.537080
en_no_macro    0.529142  0.462059  0.452327  0.504936  0.665312  0.522755

=== Expanding window: QLIKE (RV level), model x ticker ===
etf                 XLI       XLB       XLE       XLY       XLF    pooled
model                                                                    
en_full_macro  0.300235  0.279468  0.357333  0.293290  0.393131  0.324691
en_no_vix      0.394995  0.330064  0.369690  0.342665  0.461114  0.379706
en_no_macro    0.338900  0.296138  0.296345  0.297385  0.404338  0.326621
